## എൻബെഡിങ്ങുകൾ

മുമ്പത്തെ ഉദാഹരണത്തിൽ, നാം ഉയർന്ന-അതിസൂക്ഷ്മമായ ബാഗ് ഓഫ് വേർഡ് വെക്ടറുകളിൽ പ്രവർത്തിച്ചു, അവരുടെ നീളം `vocab_size` ആയിരുന്നു, കൂടാതെ നമ്മൾ സ്പർശകമായ പോസിഷണൽ പ്രതിനിധാന വെക്ടറുകളിൽ നിന്ന് സ്പാർസ് ഒന്ന്-ഹോട്ട് പ്രതിനിധാനത്തിലേക്ക് വ്യക്തമായും മാറ്റം നടത്തുകയായിരുന്നു. ഈ ഒന്ന്-ഹോട്ട് പ്രതിനിധാനം സ്മൃതി കാര്യക്ഷമമല്ല, കൂടാതെ, ഓരോ വാക്കും പരസ്പരം സ്വതന്ത്രമായി പരിഗണിക്കപ്പെടുന്നു, അതായത് ഒന്ന്-ഹോട്ട് എൻകോഡ് ചെയ്ത വെക്ടറുകൾ വാക്കുകളിലെ യഥാർത്ഥ സമാനത പ്രകടിപ്പിക്കാറില്ല.

ഈ യുണിറ്റിൽ, നാം **News AG** ഡാറ്റാസെറ്റ് കൂടുതൽ പഠനമെടുക്കും. തുടങ്ങുന്നതിനായി, നാം ഡാറ്റ ലോഡ് ചെയ്ത് മുമ്പത്തെ നോട്ട്ബുക്കിൽ നിന്നുള്ള ചില നിർവചനങ്ങൾ കണ്ടെത്ത്‌ക്കാം.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## എംബെഡിങ് എന്നതെന്താണ്?

**എംബെഡിങ്** എന്ന ആശയം വാക്കുകളെ കുറവ്-അളവിലുള്ള സാന്ദ്ര വെക്റ്ററുകളായി പ്രതിനിധാനം ചെയ്യുന്നതാണ്, ഇതുപോലെ ഏറ്റവും ശരിയായ വാക്കിന്റെ അർത്ഥം പ്രതിഫലിപ്പിക്കുന്നു. നാം പിന്നീട് അർത്ഥപൂർണമായ വേഡ് എംബെഡിംഗുകൾ എങ്ങനെ നിർമ്മിക്കാമെന്ന് ചർച്ച ചെയ്യും, എന്നാൽ ഇപ്പോൾ എംബെഡിംഗുകൾ വാക്കിന്റെ വെക്റ്ററിന്റെ അളവ് കുറയ്ക്കുന്നതിന്റെ ഒരു മാർഗമായി തിരിച്ച് ചിന്തിക്കാം.

അതുകൊണ്ട്, എംബെഡിങ് ലെയർ ഒരു വാക്കിനെ ഇൻപുട്ടായി എടുത്ത്, നിശ്ചിതമായ `embedding_size` അളവുള്ള ഔട്ട്പുട്ട് വെക്റ്റർ ഉണ്ടാക്കും. ഒരു ആശയം അര്ത്ഥത്തിൽ, ഇത് `Linear` ലെയറിൽ വളരെ സമാനമാണ്, പക്ഷേ വൺ-ഹോട്ട് എൻകോഡഡ് വെക്റ്റർ മുൻപുകളഞ്ഞ് വാക്ക് നമ്പർ ഇൻപുട്ടായി സ്വീകരിക്കാൻ ഇത് കഴിയും.

നമ്മുടെ നെറ്റ്വർക്കിലെ ആദ്യ ലെയറായി എംബെഡിങ് ലെയർ ഉപയോഗിച്ച്, നാം ബാഗ് ഓഫ് വേഡ്സ് മുതൽ **എംബെഡിങ് ബാഗ്** മോഡലിലേക്ക് മാറാം, അതിൽ ആദ്യം നമ്മുടെ വാചകത്തിലെ ഓരോ വാക്കും അനുയോജ്യമായ എംബെഡിങ്ങിലേക്ക് മാറ്റുകയും, പിന്നെ ആ എംബെഡിങ്ങുകളുടെ മുകളിൽ ചില ചേരിതിരിച്ച യൂണിയൻ ഫംഗ്ഷനുകൾ കണക്കാക്കുകയും ചെയ്യും, ഉദാഹരണങ്ങളായി `sum`, `average`, അല്ലെങ്കിൽ `max`.  

![അഞ്ചു അനുക്രമ വാക്കുകൾക്കുള്ള എംബെഡിങ് ക്ലാസിഫയർ കാണിക്കുന്ന ചിത്രം.](../../../../../translated_images/ml/embedding-classifier-example.b77f021a7ee67eee.webp)

നമ്മുടെ ക്ലാസിഫയറിന്റെ ന്യൂറൽ നെറ്റ്‌വർക്ക് എംബെഡിങ് ലെയർ দিয়ে തുടങ്ങിയ്, തുടർന്ന് അഗ്രിഗേഷൻ ലെയർ, അത്തരം ലെയറിന്മേൽ ലീനിയർ ക്ലാസിഫയർ ആയിരിക്കും:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### വ്യത്യസ്ത സീൻവഴി നീളമുളള സീൻവഴികളുമായി കൈകാര്യം ചെയ്യൽ

ഈ ആർക്കിടക്ചറിന്റെ ഫലമായി, ഞങ്ങളുടെ നെറ്റ്വർക്കിലേക്കുള്ള മിനിമി ചതുക്കൾ ഒരു നിശ്ചിത രീതിയിൽ സൃഷ്ടിക്കണം. മുമ്പത്തെ യൂണിറ്റിൽ, ബാഗ്-ഓഫ്-വർഡ്സ് ഉപയോഗിക്കുമ്പോൾ, ഒരു മിനിമി ചതുക്കിലെ എല്ലാ BoW ടെൻസറുകൾക്കും സമാനാനുപാതം ഉണ്ടായിരുന്നത് `vocab_size`, നമ്മുടെ വാചകത്തിന്റെ യാഥാർത്ഥ്യ നീളം എന്തായിരുന്നാലും. വാക്ക് എംബഡ്ഡിംഗ്സിലേക്ക് മാറുമ്പോൾ, ഓരോ വാചക സാമ്പിളിലും വ്യത്യസ്തമായ വാക്കുകളുടെ സംഖ്യ ഉണ്ടാകുമെന്നും, ആ സാമ്പിളുകൾ മിനിമി ചതുക്കുകളായി ചേർക്കുമ്പോൾ കുറച്ചുപാട് പാഡിംഗ് ഉപയോഗിക്കേണ്ടിവരുമെന്നും വരും.

ഇത് ഡാറ്റാസോഴ്സിന് `collate_fn` ഫങ്ഷൻ നൽകുന്ന അതേ സാങ്കേതിക വിദ്യ ഉപയോഗിച്ച് ചെയ്യാവുന്നതാണ്:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### എംബെഡിങ് ക്ലാസിഫയർ ട്രെയിനിംഗ്

പ്രണാൾ ദataloader സജ്ജമാക്കിയതിനുശേഷം, മുൻപ് നിർവചിച്ച ട്രെയിനിംഗ് ഫംഗ്ഷൻ ഉപയോഗിച്ച് മോഡൽ ട്രെയിൻ ചെയ്യാം:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **കുറിപ്പ്**: സമയത്തിന്റെ പരിഗണനക്ക് വേണ്ടിയാണ് ഇവിടെ 25,000 റെക്കോർഡുകൾക്ക് മാത്രം പരിശീലനം നൽകുന്നത് (ഒരു പൂർണ്ണ എപോക്കിന് താഴെ), എന്നാൽ നിങ്ങൾ തുടർച്ചയായി പരിശീലനം നടത്താം, നിരവധി എപോക്കുകൾക്ക് പരിശീലനം നൽകാനുള്ള ഒരു ഫംഗ്ഷൻ എഴുതാം, ഉയർന്ന കൃത്യത നേടാനായി ലേണിംഗ് റേറ്റ് പാരാമീറ്ററുമായി പരീക്ഷണം നടത്താം. ഏകദേശം 90% കൃത്യത വരെയെത്താൻ സാധിക്കും.


### EmbeddingBag ലെയർ ആൻഡ് വേരിയബിൾ-ലെങ്ത് സീക്വൻസ് റെപ്രസെന്റേഷൻ

മുമ്പത്തെ ആർക്കിടെക്ചറിൽ, ഓരോ സീക്വൻസും മിനിമം ബാച്ചിൽ ഫിറ്റ് ചെയ്യാൻ ഒരേ നീളത്തിലേക്ക് പാഡ് ചെയ്യേണ്ടതുണ്ടായിരുന്നു. വേരിയബിൾ ലെന്റ്Seq വാരിയബിൾ ലെംഗ്ത്സ്ക്വെൻസുകൾ പ്രതിനിധാനം ചെയ്യാനുള്ള എഫിഷ്യൻറ്റ് മാർഗ്ഗമല്ല ഇത് — മറ്റൊരു സമീപനം വേരിയബിൾ ദൈർഘ്യമുള്ള പദങ്ങളുടെയുള്ള സീക്വൻസുകൾ ഒരു വലിയ വെക്റ്ററിലുള്ള എല്ലാ സീക്വൻസുകളുടെയും ഓഫ്സെറ്റുകൾ ഉള്ള **ഓഫ്സെറ്റ്** വെക്റ്റർ ഉപയോഗിക്കുന്നതാണ്.

![ഓഫ്സെറ്റ് സീക്വൻസ് റെപ്രസെന്റേഷൻ കാണിക്കുന്ന ചിത്രം](../../../../../translated_images/ml/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **കുറിപ്പ്**: മുകളിൽ കാണിച്ചിരിക്കുന്ന ചിത്രത്തിൽ, നമ്മൾ ക്യാപ്റ്ററുകളുടെ ഒരു സീക്വൻസ് കാണിക്കുന്നു, പക്ഷേ നമ്മുടെ ഉദാഹരണത്തിൽ നാം വാക്കുകളുടെ സീക്വൻസുകളുമായി പ്രവർത്തിക്കുന്നു. എന്നിരുന്നാലും, ഓഫ്സെറ്റ് വെക്റ്റർ ഉപയോഗിച്ച് സീക്വൻസുകൾ പ്രതിനിധാനം ചെയ്യുന്നതിനുള്ള പൊതുവായ സിദ്ധാന്തം ഉപയോഗിക്കപ്പെടുന്നു.

ഓഫ്സെറ്റ് പ്രതിനിധാനത്തിൽ പ്രവർത്തിക്കാൻ, നാം [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) ലെയർ ഉപയോഗിക്കുന്നു. ഇത് `Embedding` പോലെയാണ്, പക്ഷേ ഇത് ഇരുമ്പടക്കവും ഓഫ്സെറ്റ് വെക്റ്ററും ഇൻപുട്ടായി എടുത്ത് ശരാശരിക്കലും ഉൾപ്പെടുമ്പോൾ, ശരാശരി ലെയർ അതായത് `mean`, `sum` അല്ലെങ്കിൽ `max` ആകാം.

ചര്‍ച്ച ചെയ്തിരിക്കുന്നത് `EmbeddingBag` ഉപയോഗിക്കുന്ന മാറ്റം സംഭവിച്ച നെറ്റ്‌വർക്ക് ഇവിടെ കാണാം:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

പരിശീലനത്തിനുള്ള ഡാറ്റാസെറ്റ് തയ്യാറാക്കുന്നതിനായി, ഓഫ്‌സെറ്റ് വെക്ടർ തയ്യാറാക്കുന്ന ഒരു പരിവർത്തന ഫังก്ഷൻ നൽകേണ്ടതാണ്:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

ശ്രദ്ധിക്കുക, മുന്‍ എല്ലാ ഉദാഹരണങ്ങളിലുമിപ്പോള്‍ നമ്മുടെ നെറ്റ്‌വര്‍ക്ക് രണ്ട് പാരാമീറ്ററുകള്‍ അംഗീകരിക്കുന്നു: ഡാറ്റ വെക്ടറും ഓഫ്‌സെറ്റ് വെക്ടറും, ഇവയുടെ വലുപ്പം വ്യത്യസ്തമാണ്. ഇതുപോലെ, നമ്മുടെ ഡാറ്റ ലോഡര്‍ 2-ലുപകരം 3 മൂല്യങ്ങള്‍ നല്‍കുന്നു: ടെക്സ്റ്റും ഓഫ്‌സെറ്റും വെക്ടറുകള്‍ രണ്ട് ഫീച്ചറുകളായി നല്‍കുന്നു. അതിനാല്‍, നമ്മള്‍ നമ്മുടെ പരിശീലന ഫംഗ്ഷന്‍ ಸ್ವಲ್ಪ ഇഴഞ്ഞു മാറ്റി നോക്കേണ്ടതായി വരും:  


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## സെമാന്റിക് എംബെഡിംഗുകൾ: വാക്ക്2വെക്

നമ്മുടെ മുൻകാല ഉദാഹരണത്തിൽ, മോഡൽ embedding പാളി വാക്കുകളെ വെക്ടർ പ്രതിനിധാനത്തിലേക്ക് മാപ്പ് ചെയ്യാൻ പഠിച്ചു, എന്നിരുന്നാലും, ഈ പ്രതിനിധാനം അത്ര തത്വപരമായ അർത്ഥം നൽകുന്നില്ലായിരുന്നു. സമാനമായ വാക്കുകൾ അല്ലെങ്കിൽ പദപ്രയോഗങ്ങൾ ചില വെക്ടർ ദൂരങ്ങളുടെ (ഉദാ. യൂക്ലീഡിയൻ ദൂരം) അടിസ്ഥാനത്തിൽ അടുത്തതായ വെക്ടറുകൾക്കു ചേരാൻ ഉള്ള വെക്ടർ പ്രതിനിധാനം പഠിക്കാൻ നന്നായിരിക്കും.

അതിനെ ചെയ്യാൻ, നാം embedding മോഡലിനെ ഒരു വലിയ വാചക ശേഖരത്തിൽ പ്രത്യേക രീതിയിൽ മുമ്പായി പരിശീലിപ്പിക്കേണ്ടതുണ്ട്. സെമാന്റിക് എംബെഡിംഗുകൾ പരിശീലിപ്പിക്കുന്ന ആദ്യ മാർഗങ്ങളിൽ ഒരാൾ [Word2Vec](https://en.wikipedia.org/wiki/Word2vec) എന്നാണ്. ഇത് വാക്കുകളുടെ വിതരണം പ്രതിനിധാനം സൃഷ്ടിക്കാൻ ഉപയോഗിക്കുന്ന രണ്ട് പ്രധാന ഘടനകളിലേയ്ക്ക് അടിസ്ഥാനമാക്കിയതാണ്:

 - **കൺടീന്യുസ് ബാഗ്-ഓഫ്-വേർഡ്‌സ്** (CBoW) — ഈ ഘടനയിൽ, ഓർമ്മാമെന്ന മോഡൽ പരിസരവാക്കുകളിൽ നിന്ന് ഒരു വാക്ക് പ്രവചിക്കാൻ പരിശീലിപ്പിക്കുന്നു. ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$ നല്കുമ്പോൾ, മോഡലിന്റെ ലക്ഷ്യം $(W_{-2},W_{-1},W_1,W_2)$ നിന്നു $W_0$ പ്രവചിക്കുക എന്നതാണ്.
 - **കൺടീന്യുസ് സ്കിപ്പ്-ഗ്രാം** CBoW-യ്ക്ക് വിപരീതമാണ്. മോഡൽ നിലവിലെ വാക്ക് പ്രവചിക്കാൻ പരിസരവാക്കുകളുടെ വിൻഡോ ഉപയോഗിക്കുന്നു.

CBoW വേഗമാണ്, സ്കിപ്പ്-ഗ്രാം മന്ദഗതിയിലുള്ളത്, പക്ഷേ അപൂർവമായി വരുന്ന വാക്കുകളെ പ്രതിനിധാനം ചെയ്യുന്നതിൽ മികച്ചതാണ്.

![വാക്കുകൾ വെക്ടറുകളായി മാറ്റാനുള്ള CBoW, സ്കിപ്പ്-ഗ്രാം ആൽഗോരിതങ്ങൾ കാണിക്കുന്ന ചിത്രം.](../../../../../translated_images/ml/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

ഗൂഗിൾ ന്യൂസ് ഡാറ്റാസെറ്റിൽ മുമ്പായി പരിശീലിപ്പിച്ച word2vec embedding ഉപയോഗിച്ച് പരീക്ഷിക്കാനായി, നാം **gensim** ലൈബ്രറി ഉപയോഗിക്കാം. താഴെ 'neural' എന്ന വാക്കിന്റെ ഏറ്റവും സമാനമായ വാക്കുകൾ കാണാം

> **ഗమనിക്കുക:** നിങ്ങൾ ആദ്യമായി വാക്ക് വെക്ടറുകൾ നിർമ്മിക്കുമ്പോൾ, അവ ഡൗൺലോഡ് ചെയ്യാൻ കുറച്ച് സമയം എടുക്കാവുന്നതാണ്!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


നാം വർഡിൽ നിന്നുള്ള വെക്ടർ എംബെഡിങ്ങുകളും കണക്കാക്കാവുന്നതാണ്, ക്ലാസിഫിക്കേഷൻ മോഡൽ പരിശീലനത്തിൽ ഉപയോഗിക്കാൻ (വെക്ടറിന്റെ ആദ്യം 20 ഘടകങ്ങൾ മാത്രമേ ദൃശ്യമായിരിക്കൂ):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

സെമാന്റിക്കൽ എംബഡ്ഡിംഗുകളെക്കുറിച്ചുള്ള മികച്ച കാര്യം അതായത് സേമാന്റിക്സ് മാറ്റാൻ വെക്ടർ എൻകോഡിംഗ്Manipulate ചെയ്യാനാകും. ഉദാഹരണത്തിന്, *king*യും *woman*ഉം എന്ന വാക്കുകളെ_vector_പ്രസ്തുതികളോട് όσο അടുത്ത് இருந்தോ അതുപോലെ ഒരു വാക്കും കണ്ടെത്താൻ അഭ്യർത്ഥിക്കാം, *man* എന്ന വാക്കിൽ നിന്ന് όσο ദൂരെയുള്ളതും:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

CBoW ഉം Skip-Grams ഉം “predictive” embeddings ആണ്, അവയ്ക്ക് ലൊക്കൽ കോൺടെക്സ്റ്റുകൾ മാത്രം പരിഗണനയിൽ എടുക്കുന്നുവെന്ന്. Word2Vec ആഗോള കോൺടെക്സ്റ്റിന്റെ പ്രയോജനം ഏറ്റെടുക്കാറില്ല. 

**FastText**, Word2Vec ന്റെ അടിസ്ഥാനത്തിൽ നിർമ്മിച്ചപ്പോൾ ഓരോ വാക്കിനും അതിലുള്ള അക്ഷര n-ഗ്രാമുകൾക്കും വെക്ടർ പ്രതിനിധാനങ്ങൾ പഠിക്കുന്നു. പ്രതിനിധാനങ്ങളുടെ മൂല്യങ്ങൾ ഓരോ ട്രെയിനിംഗ് ഘട്ടത്തിലും ഒരു വെക്ടറായി ശരാശരി കണക്കാക്കപ്പെടുന്നു. ഇതിലൂടെ മുൻപരിശീലനത്തിനായുള്ള കണക്കുകർക്കിൽ വളരെയധികം വർധനവുണ്ടാകുമ്പോഴും വാക്ക് ഇംബെഡിങ്ങുകൾ ഉപവാക്ക് വിവരങ്ങൾ കോഡ് ചെയ്യാൻ കഴിയുന്നു. 

മറ്റൊരു രീതി **GloVe**, സഹസംഭവ മാട്രിക്സ് ആശയം ഉപയോഗിച്ച്, സഹസംഭവ മാട്രിക്സ് ന്യുറൽ മാർഗ്ഗങ്ങൾ ഉപയോഗിച്ച് കൂടുതൽ പ്രകടനശേഷിയുള്ള, അവ്യക്താന്തര വാക്ക് വെക്ടറുകളിലേക്ക് വിഭജിക്കുന്നു.

ജെൻസിം വിവിധ വാക്ക് ഇംബെഡിങ് മോഡലുകൾ പിന്തുണയ്ക്കുന്നതുകൊണ്ട്, നിങ്ങൾ FastText, GloVe മുകളിലേക്ക് ഇംബെഡിങ്ങുകൾ മാറ്റി ഉദാഹരണവുമായി കളിക്കാം.


## PyTorch-ൽ മുൻ തയാറാക്കിയ എമ്പെഡിംഗുകൾ ഉപയോഗിക്കുന്നത്

മുകളിൽ നൽകിയ ഉദാഹരണം ഞങ്ങൾ മാറ്റിമാറ്റി, Word2Vec പോലുള്ള അർത്ഥപരമായ എമ്പെഡിംഗുകൾ ഉപയോഗിച്ച് എമ്പെഡിംഗ് ലെയറിലെ മാട്രിക്സ് മുമ്പ് തന്നെ പൂരിപ്പിക്കാം. മുൻ തയാറാക്കിയ എമ്പെഡിംഗിന്റെയും നമ്മുടെ ടെക്സ്റ്റ് കോർപസിന്റെയും വാക്കുകൾ സാധാരണയായി പൊരുത്തപ്പെടില്ല എന്ന കാര്യം ശ്രദ്ധിക്കണം, അതുകൊണ്ട് മിസ്സിംഗ് വാക്കുകൾക്കായി റാൻഡം മൂല്യങ്ങളോടെ ഭാരങ്ങൾ തയാറാക്കും:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


ഇനി നാം നമ്മളുടെ മോഡൽ പരിശീലിപ്പിക്കാം. മുമ്പത്തെ ഉദാഹരണത്തേക്കാൾ വലിയ എമ്പഡ്ഡിംഗ് ലെയർ സൈസ് കൊണ്ടും ആ Consequently, കൂടുതൽ പാരാമീറ്ററുകൾ ഉള്ളതിനാലും മോഡൽ പരിശീലിപ്പിക്കാൻ ചെലവാകുന്ന സമയം വളരെ കൂടുതലാണ്. കൂടാതെ, ഇതിന്റെ ഫലമായി, അധികം ഓവർഫിറ്റിങ്ങ് ഒഴിവാക്കാൻ ഞങ്ങൾക്ക് കൂടുതൽ ഉദാഹരണങ്ങളിൽ മോഡൽ പരിശീലിപ്പിക്കേണ്ടി വരാം.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

നമ്മുടെ കേസിൽ, വാക്കുകളുടെ വ്യത്യസ്തത കാരണം, കൃത്യതയിൽ വലിയ വർദ്ധനവ് ഉണ്ടാകുന്നില്ല.
വ്യത്യസ്ത വാക്കകോശങ്ങളുടെ പ്രശ്‌നത്തെ മറികടക്കാൻ, താഴെ പറയുന്ന പരിഹാരങ്ങളിൽ ഒന്നിനെ ഉപയോഗിക്കാം:
* നമ്മുടെ വാക്കകോശത്തിൽ word2vec മോഡൽ വീണ്ടും പരിശീലിക്കുക
* മുൻകൂട്ടി പരിശീലിച്ച word2vec മോഡലിന്റെ വാക്കകോശം ഉപയോഗിച്ച് ഡാറ്റാസെറ്റ് ലോഡ് ചെയ്യുക. ഡാറ്റാസെറ്റ് ലോഡ് ചെയ്യുമ്പോൾ ഉപയോഗിക്കേണ്ട വാക്കകോശം വ്യക്തമാക്കാം.

PyTorch `torchtext` ഫ്രെയിംവർക്ക് എംബഡ്ഡിംഗുകൾക്ക് സജ്ജമാക്കിയിരിക്കുന്നതുകൊണ്ട്, രണ്ടാമത്തെ സമീപനം വളരെ എളുപ്പമാണ്. ഉദാഹരണത്തിന്, താഴെ കാണിക്കുന്ന രീതിയിൽ GloVe അടിസ്ഥാനമാക്കിയ വാക്കകോശം സൃഷ്ടിക്കാവുന്നതാണ്:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Loaded vocabulary has the following basic operations:
* `vocab.stoi` dictionary allows us to convert word into its dictionary index
* `vocab.itos` does the opposite - converts number into word
* `vocab.vectors` is the array of embedding vectors, so to get the embedding of a word `s` we need to use `vocab.vectors[vocab.stoi[s]]`

Here is the example of manipulating embeddings to demonstrate the equation **kind-man+woman = queen** (I had to tweak the coefficient a bit to make it work):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

ആ എംബഡിംഗുകൾ ഉപയോഗിച്ച് ക്ലാസിഫയർ പരിശീലിപ്പിക്കാൻ, ആദ്യം നമുക്ക് GloVe ശബ്ദകോശം ഉപയോഗിച്ച് നമ്മുടെ ഡാറ്റാ‌സെറ്റ് എൻകോഡ് ചെയ്യേണ്ടതാണ്:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

മുകളിൽ നാം കണ്ടതുപോലെ, എല്ലാ വെക്ടർ എംബെഡ്ഡിങ്ങുകളും `vocab.vectors` മാട്രിക്‌സിലായി സംഭരിച്ചിരിക്കുന്നു. എംബെഡ്ഡിംഗ് ലെയറിന്റെ വെയ്റ്റുകളിൽ ഈ വെയ്റ്റുകൾ ലളിതമായ പകർത്തലിലൂടെ ലോഡ് ചെയ്യുന്നത് അളവുറപ്പിച്ച് എളുപ്പമാണ്:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

ഇപ്പോൾ നമുക്ക് നമ്മുടെ മോഡൽ പരിശീലിപ്പിച്ച് മെച്ചപ്പെട്ട ഫലങ്ങൾ ലഭിക്കുന്നുണ്ടോ എന്ന് നോക്കാം:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

ഞങ്ങൾക്ക് ഗണ്യമായ കൃത്യത വർധനവിനെ കാണാൻ കഴിയാത്തതിലെ ഒരു കാരണം ഞങ്ങളുടേതായ ഡാറ്റാസെറ്റിലുള്ള ചില വാക്കുകൾ മുൻകൂട്ടി പരിശീലിച്ച GloVe വാക്കുപത്തികയിൽ ഇല്ലാതാത്തത് കൊണ്ടാണ്, അതിനാൽ അവ essentially അവഗണിക്കപ്പെടുന്നു. ഈ കാര്യം മറികടക്കാൻ, ഞങ്ങൾ ഞങ്ങളുടേതായ ഡാറ്റാസെറ്റിൽ ഞങ്ങളുടെ embeddings സ്വയം പരിശീലിപ്പിക്കാം. 


## സന്ദർഭപരമായ ഉൾപ്പെടുത്തലുകൾ

Word2Vec പോലുള്ള പരമ്പരാഗത പ്രീട്രെയ്ന് ചെയ്ത എൻബെഡ്ഡിംഗ് പ്രതിനിധാനങ്ങളുടെ ഒരു പ്രധാന പരിമിതി വാക്കിന്റെ അർത്ഥ വ്യത്യാസ പ്രശ്നമാണ്. പ്രീട്രെയ്ന് ചെയ്ത എൻബെഡ്ഡിംഗുകൾ ഒരു വാക്കിന്റെ ചില അർത്ഥങ്ങൾ ഉൾക്കൊള്ളാമെങ്കിലും, ഒരു വാക്കിന്റെ എല്ലാ സാദ്ധ്യമായ അർത്ഥങ്ങളും ഒരു എൻബെഡ്ഡിംഗിൽsame encodeചെയ്യപ്പെടുന്നു. ഇത് താഴേക്ക് പോകുന്ന മോഡലുകളിൽ പ്രശ്‌നങ്ങൾ സൃഷ്ടിക്കും, കാരണം 'play' പോലുള്ള പല വാക്കുകൾ അവരുടെ ഉപയോഗ സംദർഭത്തെ ആശ്രയിച്ചുള്ള വ്യത്യസ്ത അർത്ഥങ്ങൾ ഉള്ളവയാണ്.

ഉദാഹരണമായി, താഴെ കൊടുത്തിരിക്കുന്ന രണ്ട് വ്യത്യസ്ത വാചകങ്ങളിലെ 'play' വാക്കിന്റെ അർത്ഥങ്ങൾ വളരെ വ്യത്യസ്തമാണ്:
- ഞാന്‍ ഒരു **play** നാടകം കാണാനായി തെറ്റീച്ചറിൽ പോയി.
- ജോൺ തന്റെ സുഹൃത്തുകളോടൊപ്പം **play** ചെയ്യാന്‍ ആഗ്രഹിക്കുന്നു.

മുകളിൽ നൽകിയ പ്രീട്രെയ്ന് ചെയ്ത എൻബെഡ്ഡിങ് ചുരുക്കം ഈ 'play' എന്ന വാക്കിന്റെ രണ്ടുപ്രധാന അർത്ഥങ്ങളും ഒരേ എൻബെഡ്ഡിങ്ങിൽ പ്രതിനിധീകരിക്കുന്നു. ഈ പരിമിതിയെ മറികടക്കാൻ, വാക്കുകൾ വ്യത്യസ്ത സന്ദർഭങ്ങളിൽ എങ്ങനെ ചേർക്കാവുന്നെന്നു *നോക്കുന്ന* വൻ സാങ്കേതിക കൂട്ടത്തിൽ പരിശീലിപ്പിച്ച **ഭാഷാ മോഡലിന്റെ** അടിസ്ഥാനത്തിൽ എൻബെഡ്ഡിംഗുകൾ നിർമ്മിക്കേണ്ടതുണ്ട്. ഈ ട്യൂട്ടോറിയലിൽ സന്ദർഭപരമായ എൻബെഡ്ഡിംഗുകളുടെ വിശദീകരണം ഉൾപ്പെടുന്നില്ല, പക്ഷേ അടുത്ത യൂണിറ്റിൽ ഭാഷാ മോഡലുകൾ സംബന്ധിച്ച് സംസാരിക്കുമ്പോൾ അവക്ക് തിരിച്ചുനോക്കുകയും ചെയ്യാം.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
